# 3. Rendimiento en NVIDIA Jetson Orin Nano
Ejecutar en la placa con un checkout del commit indicado por el bundle y un `.venv` compatible con su JetPack. Los datos y modelos se transfieren fuera de Git. No atribuir estas mediciones a otra placa Jetson Nano.


In [ ]:
from pathlib import Path
import platform
import sys
from shared.contract import ROOT, validate_splits
from shared.reporting.edge_bundle import verify, install
assert Path.cwd().resolve() == ROOT
print(platform.platform(), platform.machine(), sys.version)
TRANSFER = Path('/ruta/privada/al/bundle')  # Cambiar por la ruta local real
plan = verify(TRANSFER)
print(plan)


La siguiente celda copia solo archivos ausentes después de verificar hashes y el commit del checkout; nunca sobrescribe uno distinto. El bundle debe haberse creado con el notebook 02 ya validado.


In [ ]:
install(TRANSFER)
assert validate_splits()['manifest_id'] == plan['data_manifest_id']


## Paridad de inferencia antes del cronómetro
Se comparan las predicciones nuevas con las trazas del PC sobre todos los objetivos de test.


In [ ]:
import pandas as pd
from approaches.lightgbm.night_context import export_test as export_lightgbm
from approaches.lstm.adapter import export_test as export_lstm
lightgbm_run = ROOT / __import__('json').loads((TRANSFER / 'bundle_manifest.json').read_text(encoding='utf-8'))['runs']['lightgbm']
lstm_run = ROOT / __import__('json').loads((TRANSFER / 'bundle_manifest.json').read_text(encoding='utf-8'))['runs']['lstm']
for approach, produced in [('lightgbm_night', export_lightgbm(lightgbm_run)[0]), ('lstm', export_lstm(lstm_run)[0])]:
    expected = pd.read_parquet(ROOT / 'experiments' / 'runs' / 'edge_reference' / f'{approach}_test.parquet')
    actual = pd.read_parquet(produced)
    columns = ['timestamp', 'ground_truth_type', 'binary_prediction', 'stage_2_prediction', 'final_prediction']
    assert actual[columns].equals(expected[columns]), f'{approach}: la inferencia difiere del PC'
    print(approach, len(actual), 'objetivos idénticos')


## Medición
Ambos benchmarks registran plataforma, versiones, calentamiento, repeticiones, unidad y alcance. LightGBM incluye extracción causal y reglas; LSTM usa ventanas ya preparadas. Mostrar ambos alcances y no interpretar sus latencias como equivalentes.


In [ ]:
from approaches.lightgbm.night_benchmark import run_benchmark as benchmark_lightgbm
from approaches.lstm.benchmark import run_benchmark as benchmark_lstm
lightgbm_report = benchmark_lightgbm(lightgbm_run, rows=2000, repeats=5)
lstm_report = benchmark_lstm(lstm_run, rows=2000, repeats=5)
for path in (lightgbm_report, lstm_report):
    report = __import__('json').loads(path.read_text(encoding='utf-8'))
    print(path, report['prediction_unit'], report['timing_scope'], report['median_latency_ms'], report['p95_latency_ms'])


Los informes quedan en `results/reports/`, ignorados por Git. Registrar además JetPack, modo de potencia, frecuencia y temperatura durante cada medición si se usan para interpretar diferencias; no aparecen todavía en el esquema común.
